# Part 1: Acne Detection on ACNE04

This notebook trains and compares **three object detectors** on the ACNE04 dataset:

| Model | Family | Anchors |
|---|---|---|
| **YOLOv8** | One-stage CNN | Anchor-free |
| **Faster R-CNN + ResNet50-FPN** | Two-stage CNN | Anchor-based |
| **RT-DETR** (DINO family) | One-stage Transformer | Anchor-free queries |

Run end-to-end on Colab (T4 GPU) in ~3 hours.


## 0. Setup

In [ ]:
# Auto-detect how the project got into this Colab runtime.
# Supports three workflows -- you don't have to edit anything:
#   (1) git clone   - set GITHUB_URL below to a public repo URL
#   (2) zip upload  - drag acne_yang_project.zip into Colab's file pane
#   (3) Drive copy  - put the unzipped folder anywhere in your Drive
import os, glob, subprocess

GITHUB_URL = 'https://github.com/nprakash1/acne_yang_project.git'
PROJECT_DIR = 'acne_yang_project'

def _is_project_root(p):
    return os.path.isdir(os.path.join(p, 'part1_detection')) and \
           os.path.isfile(os.path.join(p, 'requirements.txt'))

# Already at project root?
if _is_project_root('.'):
    print('[setup] already at project root')
elif os.path.isdir(PROJECT_DIR) and _is_project_root(PROJECT_DIR):
    %cd $PROJECT_DIR
    print(f'[setup] cd into ./{PROJECT_DIR}')
else:
    zips = glob.glob('/content/*.zip') + glob.glob('./*.zip')
    drive_hits = glob.glob('/content/drive/MyDrive/**/part1_detection', recursive=True)
    if zips:
        z = zips[0]
        print(f'[setup] unzipping {z}')
        subprocess.check_call(['unzip', '-q', '-o', z, '-d', '/content/_extracted'])
        roots = [r for r, _, _ in os.walk('/content/_extracted') if _is_project_root(r)]
        assert roots, 'zip did not contain a project root (part1_detection/ + requirements.txt)'
        %cd $roots[0]
    elif drive_hits:
        root = os.path.dirname(drive_hits[0])
        print(f'[setup] using project from Drive: {root}')
        %cd $root
    elif GITHUB_URL:
        print(f'[setup] cloning {GITHUB_URL}')
        subprocess.check_call(['git', 'clone', GITHUB_URL])
        %cd $PROJECT_DIR
    else:
        raise RuntimeError(
            'No project source found. Either:\n'
            ' - set GITHUB_URL above to your repo, or\n'
            ' - drag acne_yang_project.zip into the Colab file pane and re-run, or\n'
            ' - mount Drive (next cell) with the project folder inside MyDrive.'
        )

print('[setup] cwd =', os.getcwd())
!pip install -q -r requirements.txt
print('[setup] dependencies installed')

In [ ]:
# Mount drive (optional, for persisting weights)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

## 1. Download ACNE04

We pull the dataset from Roboflow Universe in **both YOLOv8 and COCO formats** so the same images back all three models. You need a free Roboflow API key from https://app.roboflow.com/settings/api.

In [ ]:
import os
os.environ['ROBOFLOW_API_KEY'] = 'c1orEQIdQdZYPwUiFe2I'

from part1_detection.data import download_acne04, build_yaml
yolo_root = download_acne04(out_dir='data/acne04', fmt='yolov8')
coco_root = download_acne04(out_dir='data/acne04', fmt='coco')
yaml_path = build_yaml(yolo_root, 'data/acne04.yaml')
print('YOLO root:', yolo_root)
print('COCO root:', coco_root)
print('YAML:', yaml_path)

## 2. Train YOLOv8

In [ ]:
from part1_detection.train_ultralytics import train_yolo
yolo_results = train_yolo(data=str(yaml_path), epochs=100, imgsz=640, batch=16)

## 3. Train Faster R-CNN (ResNet50-FPN)

In [ ]:
from part1_detection.train_faster_rcnn import train as train_frcnn
frcnn_model = train_frcnn(coco_root=str(coco_root), epochs=25, batch_size=4, lr=5e-3)

## 4. Train RT-DETR (DINO family)

In [ ]:
from part1_detection.train_ultralytics import train_rtdetr
rtdetr_results = train_rtdetr(data=str(yaml_path), epochs=80, imgsz=640, batch=8)

## 5. Unified Evaluation (mAP / P / R / IoU)

All three models scored with the **same pycocotools COCOeval** for an apples-to-apples comparison.

In [ ]:
from part1_detection.evaluate import evaluate_model
import pandas as pd

metrics = {}
metrics['YOLOv8'] = evaluate_model('yolo', 'outputs/yolov8/acne/weights/best.pt', str(coco_root))
metrics['Faster R-CNN'] = evaluate_model('frcnn', 'outputs/faster_rcnn/best.pth', str(coco_root))
metrics['RT-DETR']     = evaluate_model('rtdetr', 'outputs/rtdetr/acne/weights/best.pt', str(coco_root))

df = pd.DataFrame(metrics).T[['mAP@0.5', 'mAP@0.5:0.95', 'precision@0.5', 'recall@0.5', 'mean_iou']]
df

## 6. Side-by-side bbox visualizations

In [ ]:
from part1_detection.visualize import visualize_predictions
from IPython.display import Image as IPyImage

out = visualize_predictions(
    models={
        'YOLOv8':       {'kind': 'yolo',   'weights': 'outputs/yolov8/acne/weights/best.pt'},
        'Faster R-CNN': {'kind': 'frcnn',  'weights': 'outputs/faster_rcnn/best.pth'},
        'RT-DETR':      {'kind': 'rtdetr', 'weights': 'outputs/rtdetr/acne/weights/best.pt'},
    },
    coco_root=str(coco_root),
    n_images=6,
)
IPyImage(filename=str(out))